![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 23 -- Lab 2: Garbage Photos Overfit Then Augment

Kaggle's garbage-classification photos again (TrashNet: 2,527 photos of cardboard, glass, metal, paper, plastic and trash), this time to train, and with far too few photos for a 290,000-parameter network. You write a Dataset class that caches the photos once at 64 x 64 and applies a transform every time a photo is asked for, train the deck's VGG-style network once without augmentation and once with random crops, flips, rotations and colour jitter, compare the train-against-validation gaps, test the better run once, and write a submission file.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `asdasdasasdas/garbage-classification` (about 43 MB), downloaded by `kagglehub` in the setup cell: one folder per class, split 70 / 15 / 15 into training, validation and test lists. A GPU helps (Runtime, Change runtime type, T4); on a CPU each of the two 40-epoch runs takes about ten minutes; on a GPU about a minute.

---
# Setup

In [ ]:
# kagglehub, torch, torchvision, matplotlib, pandas and scikit-learn are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q kagglehub torch torchvision matplotlib pandas scikit-learn

In [ ]:
import os
from collections import Counter
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")
root = os.path.join(path, "Garbage classification", "Garbage classification")
full = datasets.ImageFolder(root)                 # no transform: we only want the (path, label) list and the class names
CLASSES = full.classes
idx = torch.randperm(len(full), generator=torch.Generator().manual_seed(0)).tolist()
train_items = [full.samples[i] for i in idx[:1769]]
val_items = [full.samples[i] for i in idx[1769:2148]]
test_items = [full.samples[i] for i in idx[2148:]]
print(len(full), CLASSES, len(train_items), len(val_items), len(test_items))

---
# Part 1 -- The data and a Dataset class with a transform inside

The *Where It Lives* slide and Day 19's Dataset class: fixed work (open, resize) once in `__init__`, random work (the augmentation) every time in `__getitem__`.

## Task 1: Look at the data

Count the training photos per class (`Counter(label for _, label in train_items)` mapped to `CLASSES`) and show the first 8 training photos opened with `Image.open(path)` and their class names.

In [ ]:
# Your code here


## Task 2: The Dataset class

Write `class GarbageDataset(Dataset)` with `__init__(self, items, transform)` that stores `self.images = [Image.open(p).convert("RGB").resize((64, 64)) for p, _ in items]` (every photo decoded and resized once to 64 x 64), `self.labels = [label for _, label in items]` and `self.transform = transform`; `__len__`; and `__getitem__(self, i)` returning `(self.transform(self.images[i]), self.labels[i])`. Build `eval_tf = transforms.ToTensor()` and `val_ds = GarbageDataset(val_items, eval_tf)`; print `len(val_ds)` (379) and the shape of `val_ds[0][0]` (`(3, 64, 64)`).

Syntax hint (the shape of the call, not the answer):

```python
class MyData(Dataset):
    def __init__(self, items, transform):   # decode and resize once, keep the transform
    def __len__(self):                      # how many samples
    def __getitem__(self, i):               # return (transform(image_i), label_i)
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(val_ds) == 379 and tuple(x0.shape) == (3, 64, 64), 'GarbageDataset: PIL images resized to (64, 64) in __init__, transform applied in __getitem__'
print('Task 2 passed')

## Task 3: The training transform

Build `train_tf = transforms.Compose([transforms.RandomResizedCrop((64, 64), scale=(0.8, 1.0)), transforms.RandomHorizontalFlip(), transforms.RandomRotation(10), transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2), transforms.ToTensor()])` (gentle settings: the photos are small and few) and `train_aug = GarbageDataset(train_items, train_tf)`. Show the first training photo drawn 8 times (`train_aug[0][0].permute(1, 2, 0)` in a loop): the network will see a different version every epoch. Store two draws as `a, b` and check they differ.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert not torch.equal(a, b) and tuple(a.shape) == (3, 64, 64), 'train_tf: random transforms before ToTensor; two draws of the same photo must differ'
print('Task 3 passed')

## Task 4: Four loaders

Build `train_plain = GarbageDataset(train_items, eval_tf)` (the same photos, no augmentation) and four loaders: `plain_loader` and `aug_loader` over `train_plain` and `train_aug` (batch 32, `shuffle=True`), `val_loader` over `val_ds` and `test_loader` over `GarbageDataset(test_items, eval_tf)` (batch 64, no shuffle). Print the number of batches in `plain_loader` (56).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(plain_loader) == 56 and len(test_loader) == 6, 'loaders: 1,769 training photos in batches of 32; 379 test photos in batches of 64'
print('Task 4 passed')

---
# Part 2 -- The model and the two runs

The model class is Lab 1's VGG-style network with six classes, and `evaluate` is Lab 1's. The two training runs below are written out in full so that nothing is hidden: the only difference between them is the loader.

In [ ]:
def block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU())

class VGGSmall(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(
            block(3, 32), block(32, 32), nn.MaxPool2d(2),
            block(32, 64), block(64, 64), nn.MaxPool2d(2),
            block(64, 128), block(128, 128), nn.MaxPool2d(2))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, n_classes))

    def forward(self, x):
        return self.head(self.features(x))

loss_fn = nn.CrossEntropyLoss()

def evaluate(model, loader):
    model.eval()
    total, correct = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total += loss_fn(logits, yb).item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
    return total / len(loader.dataset), correct / len(loader.dataset)

print(sum(p.numel() for p in VGGSmall(6).parameters()), "parameters")

## Task 5: Run A: no augmentation

Set `EPOCHS = 40`, `torch.manual_seed(0)`, `model_a = VGGSmall(6).to(device)`, `opt = torch.optim.Adam(model_a.parameters(), lr=3e-4)`. Write the epoch loop over `plain_loader`: `model_a.train()`, then per batch move to `device`, the four moves, and count the correct `argmax` predictions so that you get the **training accuracy** of the epoch; then `evaluate(model_a, val_loader)`; append `(train_acc, val_acc)` to `hist_a`; save `model_a.state_dict()` to `best_a.pt` when the validation accuracy improves; print both accuracies. Expected: training accuracy climbing past 0.95 while validation stalls around 0.75 and jumps from epoch to epoch.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(hist_a) == 40 and hist_a[-1][0] > 0.85 and os.path.exists('best_a.pt'), 'run A: 40 epochs over plain_loader, training accuracy counted inside the loop, checkpoint on validation'
print('Task 5 passed')

## Task 6: Run B: with augmentation

The same loop, written again, with `model_b = VGGSmall(6).to(device)` (after `torch.manual_seed(0)`), a fresh optimizer (`lr=3e-4`), `aug_loader` instead of `plain_loader`, 40 epochs, history `hist_b` (training accuracy over `len(train_aug)`), checkpoint `best_b.pt`. Expected: a training accuracy that stays around 0.8 and is still climbing at the end, and a validation accuracy that catches up with run A's best.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(hist_b) == 40 and os.path.exists('best_b.pt'), 'run B: the same loop over aug_loader with a fresh model and optimizer'
print('Task 6 passed')

## Task 7: Compare the gaps

Two subplots: for run A and for run B, the training and validation accuracy against the epoch (markers, legend, titles `no augmentation` and `with augmentation`, the same y axis from 0 to 1). Then compute `gap_a` and `gap_b` as the final training accuracy minus the best validation accuracy of each run, and print them next to `best_a` and `best_b`. Expected: a gap of about 0.15 without augmentation and near 0 with it, with the two best validation accuracies close together.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert gap_b < gap_a, 'the gap: final training accuracy minus best validation accuracy; augmentation must narrow it'
print('Task 7 passed')

---
# Part 3 -- Test once, then submit

## Task 8: Test the better run

Pick the run with the higher `best_*` validation accuracy, load its checkpoint into a fresh `VGGSmall(6).to(device)` as `best`, evaluate it on `test_loader` as `test_loss, test_acc`, collect its test predictions in order into `pred_test` and the labels into `y_test`, and print the confusion matrix as a DataFrame with the class names. Which classes are confused, and does that match what you saw in the photos?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert pred_test.shape == (379,) and test_acc > 0.45, 'test: the better checkpoint, evaluated once on test_loader, predictions in loader order'
print('Task 8 passed')

## Task 9: Write the submission

Save `pd.DataFrame({"id": np.arange(379), "label": [CLASSES[p] for p in pred_test]})` to `submission.csv` without the index, read it back and print its shape and first 5 rows.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert check.shape == (379, 2) and set(check['label'].unique()) <= set(CLASSES), 'submission.csv: 379 rows, columns id and label'
print('Task 9 passed')

---
## Task 10: Reflect

Run A reached a training accuracy near 1.0 and run B did not, yet their best validation accuracies are close; explain why the lower training accuracy of run B is the plan working, and say which run you would trust on new photos and why. Run B's training accuracy was still climbing at epoch 40: what is the obvious next move, and what would you watch to know when to stop? The Dataset class resized every photo once in `__init__` but applied `train_tf` in `__getitem__`. What would go wrong if the augmentation were applied once in `__init__` instead? Which of the four transforms in `train_tf` would you remove or weaken for a dataset of handwritten digits, and which for a dataset where colour decides the class?

*Your answers here*

---
## Conclusion

- **A Dataset class with the transform inside**: photos decoded and resized once, a random transform applied every time a photo is asked for, so every epoch shows new versions.
- **Augmentation in the training transform only**: crops, flips, rotations and colour jitter for `aug_loader`; validation and test photos untouched.
- **Two runs, one loop written out**: the same VGG-style class trained without and with augmentation; the training accuracy stayed honest, the gap closed, and the validation accuracy matched the memorising run's with room to grow.
- **The hand-in**: the better checkpoint tested once, a confusion matrix, and a checked submission file.

---

Made By **Sattam Altwaim**